# Experiment 7 - Part 2: Measuring VQA Accuracy

**Course:** HAAI302COM - Artificial Intelligence Laboratory

"It seems to work" is not a result. In this notebook we score a vision model on **90 questions** (30 photos, 30 charts, 30 documents) with the metric that each benchmark uses, and then ask where and why it fails.

The three benchmarks do not judge answers the same way, because their answers are different in nature:

| Image type | Benchmark | Metric | Why |
|---|---|---|---|
| photo | VQAv2 | soft accuracy | ten people answered; agreement among them is the target |
| chart | ChartQA | relaxed accuracy | numbers read off a bar may be a little off |
| document | DocVQA | ANLS | answers are text spans; small spelling slips deserve partial credit |

Run notebook 01 first if you want to understand the `ask()` helper.

## 0. Setup

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import vqa_utils as vu

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)
df = pd.DataFrame(json.load(open("data/vqa_set.json")))
print(df.kind.value_counts().to_dict())

{'photo': 30, 'chart': 30, 'document': 30}


## 1. The three metrics on toy answers

Before trusting a number, we check the metric on cases where we know what it should say. The functions live in `vqa_utils.py`.

- `vqa_accuracy`: 1.0 if at least 3 of the 10 annotators gave the same answer, 0.33 per match below that.
- `relaxed_accuracy`: a number counts as right if it is within 5 percent of the reference.
- `anls`: 1 minus the edit distance between the two strings (divided by the longer length); it is 0 if the strings are more than half different.

In [2]:
toy = pd.DataFrame([
    ("VQA, 10 humans said 'down' 7 times", vu.vqa_accuracy("Down.", ["down"] * 7 + ["table"] * 3)),
    ("VQA, only 1 human said 'table'",     vu.vqa_accuracy("table", ["down"] * 9 + ["table"])),
    ("Chart: 68 vs reference 68",          vu.relaxed_accuracy("68", "68")),
    ("Chart: 66 vs reference 68 (3% off)", vu.relaxed_accuracy("66", "68")),
    ("Chart: 60 vs reference 68 (12% off)", vu.relaxed_accuracy("60", "68")),
    ("Chart: '68%' vs reference 68",       vu.relaxed_accuracy("68%", "68")),
    ("ANLS: 'wills lifestyle' exact",      vu.anls("Wills Lifestyle", ["wills lifestyle"])),
    ("ANLS: 'wills lifestyl' one letter missing", vu.anls("wills lifestyl", ["wills lifestyle"])),
    ("ANLS: 'price war' vs a long reference", vu.anls("price war", ["As competitor's joined the price war"])),
], columns=["case", "score"])
toy.round(3)

,case,score
0,"VQA, 10 humans said 'down' 7 times",1.000
1,"VQA, only 1 human said 'table'",0.333
2,Chart: 68 vs reference 68,1.000
3,Chart: 66 vs reference 68 (3% off),1.000
4,Chart: 60 vs reference 68 (12% off),0.000
5,Chart: '68%' vs reference 68,1.000
6,ANLS: 'wills lifestyle' exact,1.000
7,ANLS: 'wills lifestyl' one letter missing,0.933
8,ANLS: 'price war' vs a long reference,0.000


**What to notice:** the metrics behave as designed. One annotator out of ten gives only 0.33, 3 percent off a chart value still counts (1.0) but 12 percent off does not (0.0), and a missing letter costs only 0.07 in ANLS. The last row is the catch: 'price war' is a correct answer but scores 0.0 against a long reference sentence, because ANLS compares strings, not meaning. Remember this when you read document scores.

## 2. Run the model on all 90 questions

We ask every question with the short-answer instruction (the winner from notebook 01) and a system message. The calls are independent, so a `ThreadPoolExecutor` sends 4 at a time. `temperature=0` makes the run repeatable, and the disk cache means the second run is instant.

In [3]:
SYS = "You answer questions about images. Reply with only the answer: one word, one number or a short phrase. No sentences."

def run(frame, model="gemini-2.5-flash-lite", system=SYS, suffix="", max_side=None, workers=4):
    # returns a copy of frame with the model answer, tokens and per-question score added
    def one(row):
        o = vu.ask(row.image, row.question + suffix, model=model, system=system, max_side=max_side)
        return o["answer"], o["prompt_tokens"], o["completion_tokens"], o["seconds"]
    with ThreadPoolExecutor(workers) as ex:
        outs = list(ex.map(one, [r for _, r in frame.iterrows()]))
    out = frame.copy()
    out["pred"], out["p_tok"], out["c_tok"], out["sec"] = zip(*outs)
    out["score"] = [vu.score(r.kind, r.pred, r.answers, r.gold) for _, r in out.iterrows()]
    return out

base = run(df)
base[["kind", "question", "gold", "pred", "score"]].head(6)

,kind,question,gold,pred,score
0,photo,Where is he looking?,down,forward,0.000000
1,photo,What is the background metal structure?,platform,Canopy,0.000000
2,photo,What color is the court?,green,Green,1.000000
3,photo,What about this man's appearance might bother a conservative emplo...,blue hair,Purple hair,0.333333
4,photo,What is the large figure?,giraffe,Giraffes,0.333333
5,photo,What color is the door?,white,Cream,0.000000


**What to notice:** the answers are short, as asked, so the scoring is mechanical. 'Green' matches 'green' after normalisation (1.0). 'Giraffes' against 'giraffe' earns only 0.33, since just one of the ten annotators wrote the plural. 'Cream' for a door humans called 'white' earns 0.0 even though it is a defensible description. Photo questions are scored against what people said, not against what is true.

## 3. Accuracy per image type

The mean score per type is the headline number. With only 30 questions per type the estimate is noisy, so we also compute a **bootstrap interval**: resample the 30 scores with replacement 2,000 times and take the middle 95 percent of the means.

In [4]:
def boot(x, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    means = rng.choice(x, (n, len(x))).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

rows = []
for kind, g in base.groupby("kind"):
    lo, hi = boot(g.score.values)
    rows.append(dict(kind=kind, n=len(g), accuracy=g.score.mean(), ci_low=lo, ci_high=hi))
summary = pd.DataFrame(rows).set_index("kind").round(3)
summary

,n,accuracy,ci_low,ci_high
kind,,,,
chart,30,0.800,0.633,0.933
document,30,0.816,0.681,0.939
photo,30,0.633,0.467,0.789


**What to notice:** charts 0.80, documents 0.82 and photos 0.63. The 95 percent intervals are wide (about plus or minus 0.15) and the chart and document intervals overlap almost completely, so we cannot claim that one is easier than the other. Only the photo gap is suggestive, and it is the opposite of what we expected: everyday photos are hardest. The reason is the metric and the data: VQAv2 questions are open ('What color is the door?') and humans disagree, while a chart or document has one correct value.

## 4. Error analysis: why was it wrong?

A score tells us how much; it does not tell us why. We list the questions that scored below 0.5 and read them. Many "errors" are formatting differences, which a stricter prompt can fix; others are real misreadings.

In [5]:
wrong = base[base.score < 0.5][["kind", "question", "gold", "pred", "score"]].copy()
wrong["question"] = wrong.question.str[:55]
print(len(wrong), "of", len(base), "scored below 0.5")
wrong.sort_values("kind").reset_index(drop=True)

23 of 90 scored below 0.5


,kind,question,gold,pred,score
0,chart,What's the color of graph with 56 as the highest value?,Blue,Green,0.000000
1,chart,Is the starting value of the red graph the lowest value,Yes,No,0.000000
2,chart,What's the lowest value of the light blue bar?,8,5,0.000000
3,chart,"46, 43, 41 which data is missed?",47,24,0.000000
4,chart,Is the color of the middle bars gray?,Yes,No,0.000000
5,chart,What's the value of the top rightmost bar?,7,13,0.000000
6,document,Why Taco Bell's strong consumer base decreased?,As competitor's joined the price war,Price war,0.000000
7,document,what is the porosity for paper code 99103A?,12,10,0.000000
8,document,In which week does TD group show the highest diet consu,2,1,0.000000
9,document,Which group is represented by a solid curve with interm,et,TD + ET,0.000000


**What to notice:** of 23 errors, 12 are photos, 6 charts and 5 documents, and they are different in nature. Photo errors are mostly near-misses and ambiguity: 'Giraffes' for giraffe, 'In boxes' for 'in box', 'Cream' for white, 'Purple hair' for blue hair, 'six' where annotators wrote 12 (a number-word instead of a digit and also a miscount). Chart errors are real misreadings: wrong colour, yes/no flipped, 5 for 8, 13 for 7. Document errors mix a metric artifact ('Price war' against a full sentence), near-correct answers ('TD + ET' for 'et') and off-by-one reads (10 for 12, week 1 for week 2). A prompt change can fix the photo format errors but not the chart misreadings.

## 5. Parameter experiment: prompt style

We keep the model fixed and change only the instruction: **plain** (no instruction, full sentences), **short** (the system message used above), and **short + think**, which asks the model to look carefully and say only the final answer.

In [6]:
plain = run(df, system=None)
careful = run(df, system=SYS + " Look carefully at all the text and numbers in the image before answering.")
styles = pd.DataFrame({"plain": plain.groupby("kind").score.mean(),
                       "short": base.groupby("kind").score.mean(),
                       "short + look carefully": careful.groupby("kind").score.mean()}).round(3)
styles

,plain,short,short + look carefully
kind,,,
chart,0.333,0.800,0.800
document,0.525,0.816,0.825
photo,0.033,0.633,0.700


**What to notice:** the format instruction is the biggest lever of the whole notebook. With no instruction the model writes sentences and scores 0.03 on photos, 0.33 on charts and 0.53 on documents, although it often knows the answer. Adding the short-answer system message lifts all three to 0.63, 0.80 and 0.82. The extra 'look carefully' sentence changes little (photos 0.63 to 0.70 is two questions out of 30, within the noise). Prompt format affects the score far more than a vague appeal to be careful.

## 6. Parameter experiment: which model?

`gemini-2.5-flash` is the larger sibling of `flash-lite`. Is it better, and what does the extra quality cost in time and tokens? We reuse the same prompt and the same 90 questions. For `gemini-2.5-flash` we turn off the "thinking" phase (`reasoning_effort="none"`) so only the model size changes.

In [7]:
big = run(df, model="gemini-2.5-flash")
cmp = pd.DataFrame({
    "flash-lite": base.groupby("kind").score.mean(),
    "flash": big.groupby("kind").score.mean(),
    "lite_sec_per_q": base.groupby("kind").sec.mean(),
    "flash_sec_per_q": big.groupby("kind").sec.mean(),
}).round(3)
cmp

,flash-lite,flash,lite_sec_per_q,flash_sec_per_q
kind,,,,
chart,0.800,0.900,1.654,1.992
document,0.816,0.947,1.701,2.025
photo,0.633,0.744,1.830,1.817


**What to notice:** the larger model is better on every type: charts 0.80 to 0.90, documents 0.82 to 0.95, photos 0.63 to 0.74. Time per question barely changes (about 2.0 s against 1.7 s) because we turned off thinking and the answers are only a few tokens long. The larger model does not make the photo score approach the others, because many photo errors are about ambiguous questions.

## 7. Paired comparison: where do the two models disagree?

Averages hide detail. Looking at questions where one model is right and the other is wrong shows whether the bigger model fixes the small model's errors or just errs elsewhere.

In [8]:
pair = pd.DataFrame({"kind": df.kind, "lite": base.score.values, "flash": big.score.values})
pair["outcome"] = np.select([(pair.lite >= .5) & (pair.flash >= .5), (pair.lite < .5) & (pair.flash < .5),
                             (pair.lite < .5) & (pair.flash >= .5)],
                            ["both right", "both wrong", "only flash right"], "only lite right")
pd.crosstab(pair.kind, pair.outcome)

outcome,both right,both wrong,only flash right,only lite right
kind,,,,
chart,24,3,3,0
document,24,0,5,1
photo,17,9,3,1


**What to notice:** flash fixes 11 questions that flash-lite got wrong and breaks only 2. Still wrong for both: 9 photo questions and 3 chart questions, 12 in all. The photo ones are mostly the ambiguous cases from section 4, which no model can resolve, since the reference itself is one of several defensible answers. Documents benefit most: 5 fixed, 1 broken, none wrong in both.

## 8. Breakdown by question type

An average over a mixed set hides which kinds of question are easy. We label each question by the form of its reference answer (yes/no, number or text) and compute the score per image type and answer form.

In [9]:
def form(g):
    g = str(g).strip().lower()
    return "yes/no" if g in ("yes", "no") else "number" if vu._to_number(g) is not None else "text"

base["form"] = base.gold.map(form)
base.pivot_table(index="kind", columns="form", values="score", aggfunc=["mean", "count"]).round(2)

mean               count             
form     number  text yes/no number  text yes/no
kind                                            
chart      0.75  0.94   0.00   12.0  16.0    2.0
document   0.78  0.84    NaN   10.0  20.0    NaN
photo      0.50  0.47   0.91    2.0  17.0   11.0

**What to notice:** photo yes/no questions are easy (0.91 over 11 questions) while open photo questions are hard (0.47 over 17). On charts, text answers such as a country or a colour (0.94) are easier than reading a number (0.75). Chart yes/no shows 0.00 but covers only 2 questions, so it means nothing by itself. Small group sizes are the main limit of this table: use it to find where to look, then confirm with more data.

## 9. Cost and speed

Token counts times a price give the bill. We use per-million-token prices as published at the time of writing (flash-lite: 0.10 input, 0.40 output; flash: 0.30 input, 2.50 output, USD) and extrapolate to 10,000 questions. Check the current price list before relying on these numbers.

In [10]:
price = {"flash-lite": (0.10, 0.40, base), "flash": (0.30, 2.50, big)}
rows = []
for name, (pin, pout, r) in price.items():
    cost = (r.p_tok.sum() * pin + r.c_tok.sum() * pout) / 1e6
    rows.append(dict(model=name, avg_prompt_tokens=r.p_tok.mean(), avg_output_tokens=r.c_tok.mean(),
                     sec_per_q=r.sec.mean(), usd_per_10k_questions=cost / len(r) * 10000))
pd.DataFrame(rows).set_index("model").round(3)

,avg_prompt_tokens,avg_output_tokens,sec_per_q,usd_per_10k_questions
model,,,,
flash-lite,294.811,2.489,1.729,0.305
flash,294.811,2.367,1.944,0.944


**What to notice:** image tokens dominate: both models used about 295 prompt tokens per question and only about 2.4 output tokens, since the image costs a fixed ~250 tokens. flash-lite costs about 0.31 USD per 10,000 questions and flash about 0.94 USD, roughly three times as much, for 11 extra points of accuracy. At this scale both are very cheap; the choice only matters at millions of questions. Prices change, so check the current list.

In [11]:
# save every per-question result so notebook 03 and your report can reuse it
out = pd.concat([base.assign(model="flash-lite", prompt="short"), big.assign(model="flash", prompt="short"),
                 plain.assign(model="flash-lite", prompt="plain")])
out[["model", "prompt", "kind", "question", "gold", "pred", "score", "p_tok", "c_tok", "sec"]].to_csv("vqa_scores.csv", index=False)
print("saved", len(out), "rows to vqa_scores.csv")

saved 270 rows to vqa_scores.csv


## 10. Recap

- Each benchmark has its own metric; use the one that matches the answer type.
- With 30 questions per type, differences of a few points are noise; report an interval.
- Prompt format moves the score as much as the model does.
- Error analysis separates fixable formatting issues from real perception errors.

**Try yourself:** (1) lower `max_side` to 384 and see which type drops first; (2) add two example question-answer pairs to the system message (few-shot) and see if documents improve; (3) ask the model to give a confidence and check whether low confidence matches wrong answers.

Next: **Notebook 03** runs a small open-source vision model locally and a LLaVA model on a Colab GPU.